# Toxic comment classification - LSTM (Task 2)
Same dataset/split/augmentation as the RNN notebook, LSTM model instead. Target F1 >= 0.85.

In [ ]:
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
df = pd.read_csv("train.csv")
label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

X = df["comment_text"].values
y = df[label_cols].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED)

train_df = pd.DataFrame({"comment_text": X_tr})
for i, c in enumerate(label_cols):
    train_df[c] = y_tr[:, i]

test_df = pd.DataFrame({"comment_text": X_te})
for i, c in enumerate(label_cols):
    test_df[c] = y_te[:, i]

train_df.shape, test_df.shape

## Augmentation for the rare labels (same approach as RNN notebook)

In [ ]:
!pip install -q nlpaug anthropic

import nltk
for pkg in ["wordnet", "omw-1.4", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng", "punkt", "punkt_tab"]:
    nltk.download(pkg, quiet=True)

import nlpaug.augmenter.word as naw
syn_aug = naw.SynonymAug(aug_src="wordnet")

def synonym_augment(df, label, n=2):
    rows = df[df[label] == 1]
    out = []
    for _, r in rows.iterrows():
        for _ in range(n):
            t = syn_aug.augment(r["comment_text"])
            if isinstance(t, list):
                t = t[0]
            new_r = r.copy()
            new_r["comment_text"] = t
            out.append(new_r)
    return pd.DataFrame(out)

aug_threat = synonym_augment(train_df, "threat", n=3)
aug_severe = synonym_augment(train_df, "severe_toxic", n=2)
aug_identity = synonym_augment(train_df, "identity_hate", n=2)

In [ ]:
import os

USE_LLM_AUGMENT = bool(os.environ.get("ANTHROPIC_API_KEY"))

def llm_augment_label(train_df, label, n_new=400):
    import anthropic
    client = anthropic.Anthropic()
    real_examples = train_df[train_df[label] == 1]["comment_text"].sample(
        min(8, (train_df[label] == 1).sum()), random_state=SEED
    ).tolist()
    shots = "\n".join(f"- {t[:200]}" for t in real_examples)
    prompt_base = f"""You're helping build training data for a toxicity classifier.
Here are real examples labeled "{label}" from the Jigsaw dataset:
{shots}

Write NUM_PLACEHOLDER new comments with the same style, severity and variety (different topics/wording
each time, not paraphrases of each other, short informal online-comment style). One per line, no numbering."""
    lines, remaining = [], n_new
    while remaining > 0:
        batch = min(remaining, 100)
        prompt = prompt_base.replace("NUM_PLACEHOLDER", str(batch))
        msg = client.messages.create(model="claude-sonnet-4-6", max_tokens=4000,
                                      messages=[{"role": "user", "content": prompt}])
        got = [l.strip("- ").strip() for l in msg.content[0].text.strip().split("\n") if l.strip()]
        lines.extend(got)
        remaining -= len(got)
    return lines[:n_new]

if USE_LLM_AUGMENT:
    llm_frames = []
    for label, n in [("threat", 400), ("severe_toxic", 400), ("identity_hate", 400)]:
        texts = llm_augment_label(train_df, label, n_new=n)
        d = pd.DataFrame({"comment_text": texts})
        for c in label_cols:
            d[c] = 1 if c == label else 0
        llm_frames.append(d)
        print(label, ":", len(d), "new examples")
    llm_threat_df = pd.concat(llm_frames, ignore_index=True)
else:
    llm_threat_df = train_df.iloc[0:0].copy()  # empty but keeps the right dtypes

train_aug = pd.concat([train_df, aug_threat, aug_severe, aug_identity, llm_threat_df], ignore_index=True)
train_aug = train_aug.drop_duplicates(subset="comment_text").reset_index(drop=True)
train_aug[label_cols] = train_aug[label_cols].astype(int)  # concat with the (possibly empty) llm frame can upcast these to object
train_aug.shape, train_aug[label_cols].sum()

90% of the rows are all-zero (not toxic at all), which drowns out the signal for the rare labels. Undersampling the clean rows down to 3x the toxic ones helps the model actually see enough positives per batch.

In [ ]:
clean_mask = (train_aug[label_cols].sum(axis=1) == 0)
clean_rows = train_aug[clean_mask]
toxic_rows = train_aug[~clean_mask]

clean_sample = clean_rows.sample(n=min(len(clean_rows), len(toxic_rows) * 3), random_state=SEED)
train_aug = pd.concat([toxic_rows, clean_sample], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

train_aug.shape, train_aug[label_cols].sum()

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\n", " ", text)
    text = re.sub(r"\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"'", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

from nltk.tokenize import word_tokenize

train_aug["clean"] = train_aug["comment_text"].apply(clean_text)
test_df["clean"] = test_df["comment_text"].apply(clean_text)
train_aug["tokens"] = train_aug["clean"].apply(word_tokenize)
test_df["tokens"] = test_df["clean"].apply(word_tokenize)

all_words = [w for toks in train_aug["tokens"] for w in toks]
freq = Counter(all_words)

VOCAB_SIZE = 20000
word2idx = {"<PAD>": 0, "<UNK>": 1}
for w, c in freq.most_common(VOCAB_SIZE):
    word2idx[w] = len(word2idx)
VOCAB_LEN = len(word2idx)
VOCAB_LEN

## GloVe embeddings
same 100d vectors as the RNN notebook (`glove.6B.100d.txt`).

In [ ]:
EMBED_DIM = 100

def load_glove(path, word2idx, dim):
    matrix = np.random.normal(scale=0.1, size=(len(word2idx), dim)).astype("float32")
    matrix[0] = 0
    found = 0
    with open(path, encoding="utf8") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            if parts[0] in word2idx:
                matrix[word2idx[parts[0]]] = np.asarray(parts[1:], dtype="float32")
                found += 1
    print(f"found {found}/{len(word2idx)} in glove")
    return matrix

embedding_matrix = torch.tensor(load_glove("glove.6B.100d.txt", word2idx, EMBED_DIM))

In [ ]:
MAX_LEN = 60

def encode(tokens):
    return [word2idx.get(w, 1) for w in tokens]

def pad(seq, max_len=MAX_LEN):
    seq = seq[:max_len]
    return seq + [0] * (max_len - len(seq))

train_aug["ids"] = train_aug["tokens"].apply(lambda t: pad(encode(t)))
test_df["ids"] = test_df["tokens"].apply(lambda t: pad(encode(t)))

X_train = np.array(train_aug["ids"].tolist())
y_train = train_aug[label_cols].values
X_test = np.array(test_df["ids"].tolist())
y_test = test_df[label_cols].values

In [ ]:
class ToxicDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

BATCH_SIZE = 128
train_loader = DataLoader(ToxicDS(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(ToxicDS(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

pos_w = []
for c in label_cols:
    neg = (train_aug[c] == 0).sum()
    pos = (train_aug[c] == 1).sum()
    pos_w.append(neg / pos)
pos_weight = torch.clamp(torch.tensor(pos_w, dtype=torch.float32), max=5).to(device)

In [ ]:
def train_epoch(model, loader, opt, crit):
    model.train()
    total = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        out = model(xb)
        loss = crit(out, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        total += loss.item()
    return total / len(loader)

def evaluate(model, loader, threshold=0.5):
    model.eval()
    probs_all, y_all = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            probs = torch.sigmoid(model(xb))
            probs_all.append(probs.cpu().numpy())
            y_all.append(yb.numpy())
    probs_all = np.concatenate(probs_all)
    y_all = np.concatenate(y_all)
    preds = (probs_all >= threshold).astype(int)
    f1_macro = f1_score(y_all, preds, average="macro", zero_division=0)
    f1_micro = f1_score(y_all, preds, average="micro", zero_division=0)
    f1_per_label = f1_score(y_all, preds, average=None, zero_division=0)
    return f1_macro, f1_micro, f1_per_label, probs_all, y_all

def fit(model, train_loader, test_loader, opt, crit, epochs=15, patience=4):
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=2)
    best_f1, best_state, bad_epochs = 0, None, 0
    for ep in range(epochs):
        loss = train_epoch(model, train_loader, opt, crit)
        f1_macro, f1_micro, f1_per, _, _ = evaluate(model, test_loader)
        sched.step(f1_macro)
        tag = ""
        if f1_macro > best_f1:
            best_f1, best_state, bad_epochs = f1_macro, model.state_dict(), 0
            tag = " *"
        else:
            bad_epochs += 1
        print(f"ep {ep+1}: loss={loss:.4f} f1_macro={f1_macro:.4f} f1_micro={f1_micro:.4f}{tag}")
        print("  ", dict(zip(label_cols, f1_per.round(3))))
        if bad_epochs >= patience:
            print("no improvement, stopping early")
            break
    model.load_state_dict(best_state)
    return model, best_f1

class FocalLoss(nn.Module):
    def __init__(self, pos_weight, gamma=2.0):
        super().__init__()
        self.pos_weight = pos_weight
        self.gamma = gamma
    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction="none"
        )
        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        focal_term = (1 - pt) ** self.gamma
        return (focal_term * bce).mean()

## Model
2-layer bidirectional LSTM. Uses mean+max pooling over all timesteps instead of just the last hidden state - keeping only the last step throws away a lot of the sentence, this is the standard trick for this dataset.

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, embedding_matrix=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if embedding_matrix is not None:
            self.embedding.weight.data.copy_(embedding_matrix)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True,
                             num_layers=2, dropout=0.3)
        self.drop = nn.Dropout(0.4)
        self.fc = nn.Linear(hidden_dim * 4, num_classes)  # *4 = (avg+max) * 2 directions

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.lstm(emb)
        avg_pool = out.mean(dim=1)
        max_pool, _ = out.max(dim=1)
        pooled = torch.cat([avg_pool, max_pool], dim=1)
        return self.fc(self.drop(pooled))

HIDDEN_DIM = 160

lstm_model = LSTMClassifier(VOCAB_LEN, EMBED_DIM, HIDDEN_DIM, len(label_cols), embedding_matrix).to(device)
lstm_opt = optim.Adam(lstm_model.parameters(), lr=1e-3)
lstm_crit = FocalLoss(pos_weight=pos_weight, gamma=2.0)

lstm_model, lstm_f1 = fit(lstm_model, train_loader, test_loader, lstm_opt, lstm_crit, epochs=30, patience=8)
print("best f1 macro:", lstm_f1)

## Results

In [ ]:
f1_macro, f1_micro, f1_per, probs, targets = evaluate(lstm_model, test_loader)
preds = (probs >= 0.5).astype(int)
print("f1 macro:", f1_macro, "| f1 micro:", f1_micro)
print(classification_report(targets, preds, target_names=label_cols, zero_division=0))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for i, (c, ax) in enumerate(zip(label_cols, axes.ravel())):
    cm = confusion_matrix(targets[:, i], preds[:, i])
    sns.heatmap(cm, annot=True, fmt="d", ax=ax, cmap="Blues", cbar=False)
    ax.set_title(c)
    ax.set_xlabel("pred")
    ax.set_ylabel("true")
plt.tight_layout()
plt.savefig("lstm_confusion_matrices.png", dpi=150)
plt.show()

## Per-label threshold tuning

In [ ]:
from sklearn.metrics import precision_recall_curve

def best_thresholds(probs, targets):
    ths = []
    for i in range(probs.shape[1]):
        p, r, t = precision_recall_curve(targets[:, i], probs[:, i])
        f1s = 2 * p * r / (p + r + 1e-8)
        best = np.argmax(f1s)
        ths.append(t[best] if best < len(t) else 0.5)
    return ths

ths = best_thresholds(probs, targets)
tuned_preds = np.zeros_like(probs)
for i, t in enumerate(ths):
    tuned_preds[:, i] = (probs[:, i] >= t).astype(int)

print("thresholds:", dict(zip(label_cols, np.round(ths, 3))))
print("f1 macro (tuned):", f1_score(targets, tuned_preds, average="macro", zero_division=0))
print("f1 micro (tuned):", f1_score(targets, tuned_preds, average="micro", zero_division=0))

Export with outputs to PDF for the results file (File > Print > Save as PDF in Colab).